# 03. Pose Extraction — 영상에서 3D 관절 추출

**목적**: 사람 조작 영상 → 3D 관절 시퀀스 → GNN 학습 데이터(`.npz`)로 저장.

**1차 선택**: MediaPipe Hands (가벼움, 즉시 동작, 손 21 keypoints).
**2차 업그레이드**: HaMeR (Transformer 기반, 정확도 ↑) — 4번 실험 단계에서 비교.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/openvla-pose'
DATA_DIR = f'{PROJECT_ROOT}/data/poses'
os.makedirs(DATA_DIR, exist_ok=True)

!pip install -q mediapipe opencv-python

## 1. MediaPipe Hands 모델 로드

In [ ]:
import mediapipe as mp
import cv2
import numpy as np

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)
print('MediaPipe Hands ready (21 keypoints per hand, x/y/z normalized)')

## 2. 단일 이미지 테스트

In [ ]:
# 샘플 이미지 다운로드 또는 자체 업로드
TEST_IMG = f'{PROJECT_ROOT}/data/sample_hand.jpg'
if not os.path.exists(TEST_IMG):
    !wget -q -O {TEST_IMG} 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/35/Open_hand_silhouette.svg/512px-Open_hand_silhouette.svg.png' || echo 'download failed'

img_bgr = cv2.imread(TEST_IMG)
if img_bgr is not None:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = hands.process(img_rgb)
    if res.multi_hand_landmarks:
        lm = res.multi_hand_landmarks[0]
        kpts = np.array([[p.x, p.y, p.z] for p in lm.landmark])
        print('keypoints shape:', kpts.shape)
        print('first 3:\n', kpts[:3])
    else:
        print('No hand detected — 다른 샘플 이미지가 필요')
else:
    print('이미지를 못 읽음 — Drive에 sample_hand.jpg를 직접 올려주세요')

## 3. 비디오 → 시퀀스 추출 함수

In [ ]:
from typing import List, Tuple
from tqdm import tqdm

MEDIAPIPE_HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),       # thumb
    (0,5),(5,6),(6,7),(7,8),       # index
    (5,9),(9,10),(10,11),(11,12),  # middle
    (9,13),(13,14),(14,15),(15,16),# ring
    (13,17),(0,17),(17,18),(18,19),(19,20)  # pinky + palm base
]

def extract_video_hand_sequence(video_path: str, every_n_frames: int = 1):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f'Cannot open {video_path}')
    seq = []
    frame_idx = 0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    pbar = tqdm(total=n)
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % every_n_frames == 0:
            res = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if res.multi_hand_landmarks:
                kpts = np.array([[p.x, p.y, p.z] for p in res.multi_hand_landmarks[0].landmark])
            else:
                kpts = np.full((21, 3), np.nan)
            seq.append(kpts)
        frame_idx += 1
        pbar.update(1)
    cap.release(); pbar.close()
    return np.stack(seq)  # (T, 21, 3)

print('extract_video_hand_sequence ready')

## 4. 배치 추출 (Drive에 .mp4 올리고 실행)

EPIC-KITCHENS 같은 대형 데이터셋은 Colab Pro+ Drive 용량 주의.

In [ ]:
import glob
VIDEO_DIR = f'{PROJECT_ROOT}/data/videos'
os.makedirs(VIDEO_DIR, exist_ok=True)
videos = sorted(glob.glob(f'{VIDEO_DIR}/*.mp4'))
print(f'{len(videos)} videos found in {VIDEO_DIR}')

for vp in videos[:5]:
    name = os.path.splitext(os.path.basename(vp))[0]
    out = f'{DATA_DIR}/{name}.npz'
    if os.path.exists(out):
        print(f'skip (exists): {out}'); continue
    try:
        seq = extract_video_hand_sequence(vp, every_n_frames=2)
        np.savez_compressed(out,
            keypoints=seq,
            edges=np.array(MEDIAPIPE_HAND_EDGES),
            video_name=name)
        print(f'{name}: {seq.shape} → {out}')
    except Exception as e:
        print(f'{name} FAILED: {e}')

## 5. 빠른 sanity check — 시퀀스 시각화

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa

files = sorted(glob.glob(f'{DATA_DIR}/*.npz'))
if files:
    d = np.load(files[0], allow_pickle=True)
    seq, edges = d['keypoints'], d['edges']
    print('seq', seq.shape, 'edges', edges.shape)
    f = seq[len(seq)//2]
    fig = plt.figure(figsize=(5,5))
    ax = fig.add_subplot(111, projection='3d')
    valid = ~np.isnan(f).any(1)
    ax.scatter(f[valid,0], f[valid,1], f[valid,2])
    for a,b in edges:
        if valid[a] and valid[b]:
            ax.plot([f[a,0],f[b,0]], [f[a,1],f[b,1]], [f[a,2],f[b,2]], 'k-')
    ax.set_title(d['video_name'].item() if 'video_name' in d.files else 'mid frame')
    plt.show()
else:
    print('No .npz yet — Drive에 mp4 넣고 위 셀 다시 실행')